# KLTN — ABSA Token Merging · chạy full luồng trên Kaggle

Notebook này clone code từ GitHub rồi chạy `run_all.py` — toàn bộ pipeline
(train ATE → APC các biến thể → eval → hình → báo cáo).

## Trước khi chạy — bật 2 thứ trong panel bên phải

| Mục | Giá trị |
|---|---|
| **Accelerator** | `GPU T4 x2` hoặc `GPU P100` |
| **Internet** | `On` (bắt buộc — để `git clone` và tải model HuggingFace) |

## Thứ tự chạy

1. Cell **Cấu hình** → sửa `BRANCH` nếu cần
2. Cell **Clone / pull** → lấy code mới nhất
3. Cell **Cài thư viện** → chỉ lâu ở lần đầu
4. Cell **Kiểm tra môi trường** → xác nhận thấy GPU
5. Cell **SMOKE TEST** ← **chạy cái này trước**, ~5–10 phút, xác nhận cả đường ống chạy được
6. Cell **Chạy thật** → chỉ chạy khi smoke test xanh
7. Cell **Xem báo cáo** / **Đóng gói kết quả**

> **Phiên Kaggle tối đa 9–12 giờ.** Lượt đầy đủ (21 biến thể × 3 seed × 2 backbone)
> KHÔNG chạy xong trong một phiên. Hãy dùng `--resume` và chia nhiều phiên, hoặc
> thu hẹp phạm vi ở cell **Chạy thật**.

## 1 · Cấu hình

In [ ]:
import os
from pathlib import Path

# ── Nguồn code ────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/hotuyen21pt/KLTN-Token-Merging.git"
BRANCH    = "tuyen"
REPO_NAME = "KLTN-Token-Merging"

# Repo private? Thêm Kaggle Secret tên GITHUB_TOKEN (Add-ons → Secrets).
# Repo public thì bỏ qua, cell clone tự chạy không cần token.
USE_TOKEN = False

# ── Thư mục làm việc ──────────────────────────────────────────────────────
# /kaggle/working được giữ lại sau phiên (giới hạn ~20 GB).
# /kaggle/temp bị xoá khi hết phiên → để cache model cho khỏi tốn quota output.
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR  = WORK_ROOT / REPO_NAME
CACHE_DIR = Path("/kaggle/temp/hf") if Path("/kaggle").exists() else WORK_ROOT / ".hf"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["MPLBACKEND"] = "Agg"   # Kaggle không có màn hình

print(f"Repo dir  : {REPO_DIR}")
print(f"HF cache  : {CACHE_DIR}")
print(f"Branch    : {BRANCH}")

## 2 · Clone / pull code từ GitHub

Chạy được nhiều lần:

- **Lần đầu** → `git clone`
- **Lần sau** → sửa lại `origin`, `git fetch --all --prune`, rồi `git reset --hard origin/<BRANCH>`

`reset --hard` đảm bảo code trong phiên khớp *chính xác* với GitHub — mọi thay đổi
cục bộ trong `/kaggle/working` sẽ bị bỏ. Kết quả thực nghiệm nằm ngoài vùng git
theo dõi (đã `.gitignore`) nên **không** bị xoá; muốn chắc thì chạy cell đóng gói
ở cuối trước khi pull lại.

In [ ]:
import subprocess, sys, shutil

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh, in cả stdout lẫn stderr theo thời gian thực."""
    print(f"$ {' '.join(cmd)}")
    proc = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {proc.returncode}): {' '.join(cmd)}")
    return proc.returncode


# ── Dựng URL (kèm token nếu repo private) ─────────────────────────────────
clone_url = REPO_URL
if USE_TOKEN:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    print("Dùng GITHUB_TOKEN từ Kaggle Secrets")


if (REPO_DIR / ".git").is_dir():
    print(f"Đã có repo tại {REPO_DIR} → cập nhật\n")
    sh(["git", "remote", "set-url", "origin", clone_url], cwd=REPO_DIR)
    sh(["git", "fetch", "--all", "--prune", "--tags"], cwd=REPO_DIR)
    sh(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
    # Dọn file rác do git tạo, GIỮ LẠI kết quả (đã nằm trong .gitignore)
    sh(["git", "clean", "-fd"], cwd=REPO_DIR)
else:
    print(f"Chưa có repo → clone nhánh {BRANCH}\n")
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--branch", BRANCH, "--single-branch", clone_url, str(REPO_DIR)])

# Giấu token khỏi .git/config sau khi xong
if USE_TOKEN:
    sh(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print()
sh(["git", "log", "--oneline", "-5"], cwd=REPO_DIR)
sh(["git", "status", "--short", "--branch"], cwd=REPO_DIR)
print(f"\nThư mục hiện tại: {os.getcwd()}")

## 3 · Cài thư viện

Image của Kaggle đã có sẵn `torch`, `transformers`, `scikit-learn`, `matplotlib`,
`pandas`. Cell này chỉ bù những gói còn thiếu (`pyabsa`, `python-Levenshtein`, …).
Đặt `FULL_INSTALL = True` nếu muốn cài đúng theo `requirements.txt` (lâu hơn nhiều
và có thể đụng phiên bản torch của Kaggle).

In [ ]:
FULL_INSTALL = False

if FULL_INSTALL:
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
else:
    sh([sys.executable, "-m", "pip", "install", "-q",
        "python-Levenshtein>=0.25.0", "pyabsa>=2.4.0,<3", "seaborn", "tqdm"],
       check=False)

for pkg in ["torch", "transformers", "sklearn", "matplotlib", "seaborn", "Levenshtein"]:
    try:
        m = __import__(pkg)
        print(f"  {pkg:<14}: {getattr(m, '__version__', 'ok')}")
    except Exception as e:
        print(f"  {pkg:<14}: THIẾU ({type(e).__name__})")

## 4 · Kiểm tra môi trường

`run_all.py --list` in ra 12 stage và 21 biến thể; `--stages env` ghi snapshot
môi trường ra `reports/00_environment.txt`.

In [ ]:
sh(["nvidia-smi"], check=False)
print()
sh([sys.executable, "run_all.py", "--list"])

## 5 · SMOKE TEST — chạy cái này trước

`--smoke` dựng một dataset tí hon (48 train / 16 dev / 16 test) rồi chạy **toàn bộ**
đường ống với 1 epoch, 1 seed, 3 biến thể (phủ đủ 3 nhánh code: post-ToMe resize,
post-ToMe compact, pre-ToMe).

Mọi output đi vào `smoke_run/` — **không đè lên kết quả thật**.

Mất khoảng **5–10 phút** trên GPU (phần lớn là tải `t5-base` + `bert-base-uncased`
lần đầu). Nếu cell này xanh thì lượt đầy đủ cũng sẽ chạy được.

In [ ]:
rc = sh([sys.executable, "run_all.py", "--smoke", "--skip", "uos"], check=False)
print(f"\n{'='*70}")
print("SMOKE TEST: " + ("ĐẠT — đường ống chạy được" if rc == 0 else f"HỎNG (exit {rc})"))
print("=" * 70)
print("Log chi tiết : smoke_run/reports/logs/<stage>.log")
print("Báo cáo      : smoke_run/reports/REPORT.md")

In [ ]:
# Bảng trạng thái từng stage của lượt smoke
import json

status_file = REPO_DIR / "smoke_run" / "reports" / "run_status.json"
if status_file.is_file():
    data = json.loads(status_file.read_text(encoding="utf-8"))
    print(f"{'stage':<12} {'trạng thái':<10} {'phút':>6}  ghi chú")
    print("-" * 78)
    for s in data["stages"]:
        print(f"{s['stage']:<12} {s['status']:<10} "
              f"{s.get('duration_sec', 0)/60:6.1f}  {s.get('note', '')[:44]}")
else:
    print("Chưa có smoke_run/reports/run_status.json — chạy cell smoke test trước.")

## 6 · Chạy thật

Chỉ chạy khi smoke test đã xanh. Sửa `ARGS` cho hợp với thời gian phiên:

| Phạm vi | `ARGS` | Ước lượng |
|---|---|---|
| Rất gọn | `--seeds 42 --backbones bert --variants resize --skip uos gas` | vài giờ |
| Vừa | `--seeds 42 --variants resize compact --skip uos` | ~1 phiên |
| Đầy đủ | `--skip uos` | nhiều phiên, cần `--resume` |

Ghi chú:

- `--resume` **luôn nên bật** để chạy tiếp ở phiên sau.
- `uos` cần Ollama chạy tại `localhost:11434` — Kaggle không có nên luôn bỏ qua;
  không bỏ thì stage tự skip kèm cảnh báo.
- Repo **không** chứa checkpoint (đã `.gitignore`) nên stage `ate` sẽ train T5 ATE
  từ đầu. Đó là điều kiện bắt buộc cho `ate_infer` và mọi eval end-to-end.

In [ ]:
ARGS = [
    "--seeds", "42",
    "--backbones", "bert",
    "--variants", "resize",
    "--skip", "uos",
    "--resume",
]

# Xem trước sẽ chạy những lệnh gì (không tốn GPU)
sh([sys.executable, "run_all.py", *ARGS, "--dry-run"])

In [ ]:
# ⚠️ Cell này chạy rất lâu. Kiểm tra lại ARGS ở cell trên trước khi chạy.
rc = sh([sys.executable, "run_all.py", *ARGS], check=False)
print(f"\nexit code: {rc}")

## 7 · Xem báo cáo

In [ ]:
from IPython.display import Markdown, display

# Đổi thành "smoke_run/reports" để xem báo cáo của lượt smoke
REPORT_DIR = REPO_DIR / "reports"

report = REPORT_DIR / "REPORT.md"
if report.is_file():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print(f"Chưa có {report} — chạy `run_all.py --stages report` trước.")

In [ ]:
# Các bảng luận văn (nếu stage multiseed đã chạy)
tables = REPO_DIR / "runs_multiseed" / "thesis_tables.txt"
if tables.is_file():
    print(tables.read_text(encoding="utf-8"))
else:
    print(f"Chưa có {tables}")

## 8 · Đóng gói kết quả để tải về

Gom báo cáo + CSV + hình (KHÔNG gồm checkpoint `.pt`, quá nặng) thành một file zip
trong `/kaggle/working` để tải về từ tab Output.

In [ ]:
import zipfile
from datetime import datetime

PATTERNS = [
    "reports/**/*",
    "smoke_run/reports/**/*",
    "runs_ate/*.csv", "runs_ate/*.txt",
    "runs_joint/experiment_results_joint.*",
    "runs_joint_t5/experiment_results_joint.*",
    "runs_multiseed/*.csv", "runs_multiseed/*.txt",
    "runs_bert_gold/**/*.csv",
    "runs_gas/*.json", "runs_gas/*.csv",
    "thesis/figures/*.png", "thesis/figures/*.pdf",
]

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path("/kaggle/working") / f"kltn_results_{stamp}.zip" \
    if Path("/kaggle/working").exists() else REPO_DIR / f"kltn_results_{stamp}.zip"

n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for pat in PATTERNS:
        for f in sorted(REPO_DIR.glob(pat)):
            if f.is_file() and f.suffix not in {".pt", ".bin", ".safetensors"}:
                zf.write(f, f.relative_to(REPO_DIR))
                n += 1

print(f"Đã đóng gói {n} file → {zip_path}")
print(f"Dung lượng: {zip_path.stat().st_size / 1024:.1f} KB")
print("\nTải về ở tab Output (panel bên phải) sau khi Save Version.")